# Graph Manager
> Store expanded triples either in a **pure-Python in-memory graph** (default on low-power runtime) or an **rdflib Graph/Dataset** when available

In [ ]:
#| default_exp core.graph

## imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Tuple, Iterable

from cogitarelink.core.debug import get_logger
log = get_logger("graph")

# optional rdflib
try:
    from rdflib import Graph, ConjunctiveGraph, URIRef, Dataset
    _HAS_RDFLIB = True
except ModuleNotFoundError:
    _HAS_RDFLIB = False

## backend protocol

In [ ]:
#| export
class GraphBackend:
    "Backend protocol – concrete classes must implement these methods."

    def add_nquads(self, nquads: str, graph_id: str | None = None) -> None: ...
    def triples(self, subj=None, pred=None, obj=None) -> Iterable[Tuple]: ...
    def size(self) -> int: ...
    def add_named_graph(self, graph_id: str, nquads: str): ...

## pure Python backend  

In [ ]:
#| export
class InMemoryGraph(GraphBackend):
    "Very small, set-based triple store (no indexes)."
    def __init__(self):
        self._triples: set[Tuple] = set()

    def add_nquads(self, nquads: str, graph_id: str | None = None) -> None:
        for line in nquads.strip().splitlines():
            parts = line.rstrip(" .").split(" ", 3)
            if len(parts) == 3: # Handle N-Triples like format S P O
                s, p, o = parts
                self._triples.add((s, p, o))
            elif len(parts) == 4: # Handle N-Quads format S P O G
                s, p, o, _g = parts # Ignore graph component (_g)
                self._triples.add((s, p, o))
            else:
                log.warning(f"Skipping malformed line: {line}")


    def triples(self, subj=None, pred=None, obj=None):
        for s, p, o in self._triples:
            if subj and subj != s:   continue
            if pred and pred != p:   continue
            if obj  and obj  != o:   continue
            yield s, p, o

    def size(self): return len(self._triples)
    def add_named_graph(self, graph_id, nquads): self.add_nquads(nquads)

## rdflib backend  

In [ ]:
#| export
if _HAS_RDFLIB:
    class RDFLibGraph(GraphBackend):
        "Delegate to rdflib Dataset (quad store)."
        def __init__(self):
            self.ds = Dataset()

        def add_nquads(self, nquads: str, graph_id: str | None = None) -> None:
            self.ds.parse(data=nquads, format="nquads")

        def triples(self, subj=None, pred=None, obj=None):
            q = self.ds.quads((subj, pred, obj, None))
            for s, p, o, _ctx in q: yield s, p, o

        def size(self): return len(self.ds)
        def add_named_graph(self, graph_id, nquads):
            g = self.ds.graph(graph_id)
            g.parse(data=nquads, format="nquads")

## GraphManager

In [ ]:
#| export
class GraphManager:
    """
    Facade that auto-selects backend.

    Parameters
    ----------
    use_rdflib : bool | 'auto'
        * 'auto'  → choose rdflib when import succeeded.
        * True    → require rdflib else RuntimeError.
        * False   → force in-memory backend.
    """

    def __init__(self, use_rdflib: bool | str = "auto"):
        if use_rdflib == "auto":
            use_rdflib = _HAS_RDFLIB
        if use_rdflib:
            if not _HAS_RDFLIB:
                raise RuntimeError("rdflib not available")
            self._backend: GraphBackend = RDFLibGraph()          # type: ignore
        else:
            self._backend = InMemoryGraph()

    # ------------------------------------------------------------------
    # proxy helpers
    # ------------------------------------------------------------------
    def ingest_nquads(self, nquads: str): self._backend.add_nquads(nquads)
    def query(self, subj=None, pred=None, obj=None):
        return list(self._backend.triples(subj, pred, obj))
    def size(self): return self._backend.size()
    def ingest_entity(self, ent:"Entity"):
        self.ingest_nquads(ent.normalized())
        for child in ent.children:
            gid=f"{ent.id}#child"
            self._backend.add_named_graph(gid, child.normalized())

## quick tests  

In [ ]:
#| hide
from fastcore.test import test_ne, test_eq
gm = GraphManager(use_rdflib=False)
gm.ingest_nquads('<a> <b> "c" .')
test_ne(gm.size(), 0) # Check that size is not zero
test_eq(len(gm.query(subj="<a>")), 1)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()